In [1]:
import pandas as pd
import numpy as np
import math
import random

from sklearn.cluster import HDBSCAN

random.seed(11)
np.random.seed(11)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_DIR = "/kaggle/input/datasets/szishanali/"  # <-- change if needed
import os
if not os.path.exists(f"{DATA_DIR}/user-data/users (1).csv"):
    DATA_DIR = "."  # fallback: files sitting next to the notebook

users = pd.read_csv(f"{DATA_DIR}/user-data/users (1).csv")
tweets = pd.read_csv(f"{DATA_DIR}tweets-single-lingual/tweets (1).csv")

print(users.shape, tweets.shape)
users.head()

GEO = {
    ("London", None):            (51.5074, -0.1278),
    ("London", "North London"):  (51.5590, -0.1425),
    ("London", "West London"):   (51.5074, -0.2200),
    ("London", "East London"):   (51.5450, -0.0350),
    ("Manchester", None):        (53.4808, -2.2426),
    ("Manchester", "North Manchester"): (53.5100, -2.2300),
    ("Manchester", "South Manchester"): (53.4400, -2.2500),
    ("New York", None):          (40.7128, -74.0060),
    ("New York", "Manhattan"):   (40.7831, -73.9712),
    ("New York", "Brooklyn"):    (40.6782, -73.9442),
    ("Los Angeles", None):       (34.0522, -118.2437),
    ("Los Angeles", "Downtown LA"): (34.0407, -118.2468),
    ("Los Angeles", "Hollywood"):   (34.0928, -118.3287),
    ("Chicago", None):           (41.8781, -87.6298),
    ("Chicago", "South Side"):   (41.7508, -87.5960),
}

users["home_lat"] = users.apply(lambda r: GEO[(r["location"], r["locality"])][0], axis=1)
users["home_lon"] = users.apply(lambda r: GEO[(r["location"], r["locality"])][1], axis=1)
users_idx = users.set_index("user_id").to_dict("index")

LOCALITY_TO_CITY = {u["locality"]: u["location"] for _, u in users.iterrows()}
LOCALITIES_SORTED = sorted(LOCALITY_TO_CITY.keys(), key=len, reverse=True)
EXTRACTION_MISS_RATE = 0.08  # simulated NLP extraction noise

def extract_location_from_text(text):
    for loc in LOCALITIES_SORTED:
        if loc.lower() in text.lower():
            if random.random() < EXTRACTION_MISS_RATE:
                continue
            city = LOCALITY_TO_CITY[loc]
            lat, lon = GEO[(city, loc)]
            return city, loc, lat, lon
    return None, None, None, None

ext_city, ext_loc, ext_lat, ext_lon, used_fallback = [], [], [], [], []
for _, t in tweets.iterrows():
    c, l, lat, lon = extract_location_from_text(t["tweet_text"])
    if lat is None:
        u = users_idx[t["user_id"]]
        c, l, lat, lon = u["location"], u["locality"], u["home_lat"], u["home_lon"]
        used_fallback.append(True)
    else:
        used_fallback.append(False)
    ext_city.append(c); ext_loc.append(l); ext_lat.append(lat); ext_lon.append(lon)

tweets["feat_city"] = ext_city
tweets["feat_locality"] = ext_loc
tweets["feat_lat"] = ext_lat
tweets["feat_lon"] = ext_lon
tweets["location_from_fallback"] = used_fallback  # True = no place name in text -> used tweeting/home location

tweets["hour"] = pd.to_datetime(tweets["timestamp"]).dt.hour
tweets["is_weekend"] = (tweets["day_type"] == "Weekend").astype(int)

print("Located from text:", (~tweets['location_from_fallback']).sum(),
      "| Fell back to tweeting location:", tweets['location_from_fallback'].sum())
EARTH_R_KM = 6371.0
HOUR_KM_PER_HOUR = 4.0       # max circular hour diff (12h) -> 48 km penalty
WEEKEND_MISMATCH_KM = 15.0   # weekday vs weekend tweet -> 15 km penalty

def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    return 2 * EARTH_R_KM * math.asin(math.sqrt(a))

def build_distance_matrix(sub):
    n = len(sub)
    lat = sub["feat_lat"].values; lon = sub["feat_lon"].values
    hour = sub["hour"].values; wknd = sub["is_weekend"].values
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d_geo = haversine_km(lat[i], lon[i], lat[j], lon[j])
            hd = abs(int(hour[i]) - int(hour[j])); hd = min(hd, 24 - hd)
            d_time = HOUR_KM_PER_HOUR * hd
            d_day = WEEKEND_MISMATCH_KM if wknd[i] != wknd[j] else 0.0
            D[i, j] = D[j, i] = d_geo + d_time + d_day
    return D

tweets["context_cluster"] = -1
cluster_summaries = []

for ev in tweets["broad_event"].unique():
    mask = tweets["broad_event"] == ev
    idx = tweets.index[mask]
    sub = tweets.loc[idx]
    n = len(idx)
    D = build_distance_matrix(sub)
    min_cluster_size = max(3, n // 12)
    clusterer = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=2, metric="precomputed")
    labels = clusterer.fit_predict(D)
    tweets.loc[idx, "context_cluster"] = labels

    for lab in sorted(set(labels)):
        csub = sub[labels == lab]
        cluster_summaries.append({
            "broad_event": ev, "cluster": lab, "size": len(csub),
            "centroid_lat": round(csub["feat_lat"].mean(), 4),
            "centroid_lon": round(csub["feat_lon"].mean(), 4),
            "dominant_city": csub["feat_city"].mode().iat[0] if not csub["feat_city"].mode().empty else None,
            "dominant_locality": csub["feat_locality"].mode().iat[0] if lab != -1 and not csub["feat_locality"].mode().empty else "(noise / mixed)",
            "dominant_slot": csub["slot"].mode().iat[0],
            "dominant_day_type": csub["day_type"].mode().iat[0],
        })

tweets["context_id"] = tweets.apply(
    lambda r: f"{r['broad_event']}__C{r['context_cluster']}" if r["context_cluster"] != -1 else f"{r['broad_event']}__generic",
    axis=1
)

cluster_summary_df = pd.DataFrame(cluster_summaries).sort_values(["broad_event", "cluster"]).reset_index(drop=True)
centroid_lookup = cluster_summary_df.set_index(
    cluster_summary_df.apply(lambda r: f"{r['broad_event']}__C{r['cluster']}" if r['cluster'] != -1 else f"{r['broad_event']}__generic", axis=1)
)[["centroid_lat", "centroid_lon", "dominant_slot", "dominant_day_type"]].to_dict("index")

print(f"{(cluster_summary_df['cluster'] != -1).sum()} genuine context clusters discovered "
      f"across {tweets['broad_event'].nunique()} broad categories")
print(f"{(tweets['context_cluster'] == -1).sum()} / {len(tweets)} tweets flagged as noise (no coherent local context)")
cluster_summary_df
USER_EVENTS = {
    "U1":  ["Fire", "Sports", "Election"], "U2":  ["Fire", "Protest", "Viral Disease"],
    "U3":  ["Road Accident", "Sports", "Election"], "U4":  ["Earthquake", "Fire", "Sports"],
    "U5":  ["Earthquake", "Protest", "Election"], "U6":  ["Road Accident", "Viral Disease", "Sports"],
    "U7":  ["Road Accident", "Protest", "Election"], "U8":  ["Protest", "Fire", "Sports"],
    "U9":  ["Protest", "Viral Disease", "Election"], "U10": ["Viral Disease", "Road Accident", "Sports"],
}

user_profiles = {}
for uid, u in users.set_index("user_id").iterrows():
    hist = {}
    own_tweets = tweets[tweets["user_id"] == uid]
    for ev in USER_EVENTS[uid]:
        ev_tweets = own_tweets[own_tweets["broad_event"] == ev]
        if len(ev_tweets):
            hist[ev] = {
                "lat": ev_tweets["feat_lat"].mean(), "lon": ev_tweets["feat_lon"].mean(),
                "slot": ev_tweets["slot"].mode().iat[0], "day_type": ev_tweets["day_type"].mode().iat[0],
            }
    user_profiles[uid] = {"home_lat": u["home_lat"], "home_lon": u["home_lon"],
                           "interests": USER_EVENTS[uid], "history": hist}

user_profiles["U1"]
def is_relevant(row, uid):
    prof = user_profiles[uid]
    if row["broad_event"] not in prof["interests"]:
        return False
    if row["mentions_location_in_text"]:
        u = users.set_index("user_id").loc[uid]
        return row["event_context_city"] == u["location"] and row["event_context_locality"] == u["locality"]
    return True
DIST_THRESHOLD_KM = 25.0  # tune: max real-world radius treated as "the same local context"

def baseline_recommend(row, uid):
    return row["broad_event"] in user_profiles[uid]["interests"]

def framework_recommend(row, uid):
    prof = user_profiles[uid]
    if row["broad_event"] not in prof["interests"]:
        return False, 0.0
    if row["location_from_fallback"] or row["context_cluster"] == -1:
        return True, 0.5
    c = centroid_lookup[row["context_id"]]
    hist = prof["history"].get(row["broad_event"])
    ref_lat, ref_lon = (hist["lat"], hist["lon"]) if hist else (prof["home_lat"], prof["home_lon"])
    d = haversine_km(c["centroid_lat"], c["centroid_lon"], ref_lat, ref_lon)
    if d > DIST_THRESHOLD_KM:
        return False, 0.0
    confidence = 0.7
    if hist and c["dominant_slot"] == hist["slot"]:
        confidence += 0.15
    if hist and c["dominant_day_type"] == hist["day_type"]:
        confidence += 0.15
    return True, round(min(confidence, 1.0), 2)
all_users = list(user_profiles.keys())
records = []
for _, t in tweets.iterrows():
    for uid in all_users:
        if uid == t["user_id"]:
            continue
        gt = is_relevant(t, uid)
        base = baseline_recommend(t, uid)
        fw, conf = framework_recommend(t, uid)
        records.append({"tweet_id": t["tweet_id"], "author": t["user_id"], "candidate_user": uid,
                         "broad_event": t["broad_event"], "context_id": t["context_id"],
                         "ground_truth_relevant": gt, "baseline_recommends": base,
                         "framework_recommends": fw, "framework_confidence": conf})

res = pd.DataFrame(records)

def prf(df, col):
    tp = ((df[col]) & (df["ground_truth_relevant"])).sum()
    fp = ((df[col]) & (~df["ground_truth_relevant"])).sum()
    fn = ((~df[col]) & (df["ground_truth_relevant"])).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return pd.Series({"recommended": int(df[col].sum()), "true_positives": int(tp), "false_positives": int(fp),
                       "false_negatives": int(fn), "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)})

summary = pd.DataFrame({"Baseline (broad-category only)": prf(res, "baseline_recommends"),
                         "HDBSCAN Context-Aware Framework": prf(res, "framework_recommends")}).T
summary.index.name = "Recommender"
summary


(10, 7) (300, 18)
Located from text: 194 | Fell back to tweeting location: 106
40 genuine context clusters discovered across 7 broad categories
13 / 300 tweets flagged as noise (no coherent local context)


,recommended,true_positives,false_positives,false_negatives,precision,recall,f1
Recommender,,,,,,,
Baseline (broad-category only),1080.0,331.0,749.0,0.0,0.306,1.0,0.469
HDBSCAN Context-Aware Framework,467.0,331.0,136.0,0.0,0.709,1.0,0.830
